# Audit evidence: Adult income and multiple comparisons

## Question, target, and record scope

The submitted question asks which recorded characteristics are associated with income above $50,000 in the Adult extract. The independent loader returns 48,842 records, of which 11,687 (23.93%) are in the `>50K` class. This is a descriptive association question; it does not identify causal effects. The submitted analyses are unweighted.

## Submitted tests and multiplicity

The saved code conducts six numeric-feature Mann–Whitney tests and eight categorical-feature chi-square tests, for a family of 14 feature-level hypotheses. The same family is passed to Benjamini–Hochberg and Bonferroni adjustment. Thirteen features are significant under each correction; `fnlwgt` is not. Ten raw p-values are returned by SciPy as floating-point 0.0 and printed as `0`. This is numeric underflow, not a literal zero probability; threshold decisions are unchanged.

The submitted effect table sorts Cramer's V for categorical features and absolute rank-biserial effects for numeric features together. Those measures have different definitions and are not calibrated to support a single cross-type ranking. The evidence table identifies each metric by name.

## Missingness and model reproduction

Missing workclass and occupation records have high-income rates near 9.5%, compared with 24.8% for observed records. Missing native-country records have a slightly higher rate (25.7% vs 23.9%), but the Newcombe 95% interval for that difference includes zero. Therefore the blanket statement that all missing values skew toward lower income is not supported.

The submitted stratified random 75/25 logistic-model AUC is independently reproduced. It is an internal split of the same extract, not external validation. The code below displays aggregate metrics only; it does not display source rows.

## Evidence code

Run the code cell to regenerate the independent audit. Persisted findings are in `results.json`; the narrative report is `audit.md`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'audit_tools.py').is_file())
sys.path.insert(0, str(project_root / 'src'))
from adult_audit import audit_adult_income

audit = audit_adult_income()
features = audit['feature_associations']['results_sorted_by_reported_effect']
feature_table = pd.DataFrame([
    {
        'feature': row['feature'],
        'type': row['type'],
        'effect metric': row['effect_name'],
        'effect': row['effect'],
        'p-BH < .05': row['p_bh'] < 0.05,
        'p-Bonferroni < .05': row['p_bonferroni'] < 0.05,
        'raw p-value': '0.0 (underflow)' if row['p_value'] == 0 else f"{row['p_value']:.3g}",
    }
    for row in features
])
display(feature_table.round({'effect': 3}))

missingness_table = pd.DataFrame([
    {
        'feature': feature,
        'missing rate': summary['missing']['rate'],
        'observed rate': summary['observed']['rate'],
        'difference (pp)': 100 * summary['missing_minus_observed_income_rate'],
        'Newcombe 95% CI (pp)': tuple(round(100 * x, 2) for x in summary['difference_ci_95']),
        'missing n': summary['missing']['n'],
    }
    for feature, summary in audit['missingness'].items()
])
display(missingness_table.round({'missing rate': 4, 'observed rate': 4, 'difference (pp)': 2}))

print({
    'n': audit['n_total'],
    'high_income_prevalence': audit['positive_prevalence'],
    'fnlwgt_weighted_prevalence_descriptive_only': audit['fnlwgt_weighted_prevalence'],
    'feature_tests': audit['feature_associations']['n_feature_level_tests'],
    'BH_significant': audit['feature_associations']['bh_significant_count'],
    'Bonferroni_significant': audit['feature_associations']['bonferroni_significant_count'],
    'AUC': audit['adjusted_model']['test_auc'],
    'AUC_bootstrap_95_CI': audit['adjusted_model']['test_auc_ci_95'],
    'saved_run_exit_code': audit['saved_ai_run']['exit_code'],
})

,feature,type,effect metric,effect,p-BH < .05,p-Bonferroni < .05,raw p-value
0,relationship,categorical,Cramer's V,0.454,True,True,0.0 (underflow)
1,marital-status,categorical,Cramer's V,0.448,True,True,0.0 (underflow)
2,education-num,numeric,absolute rank-biserial,0.432,True,True,0.0 (underflow)
3,education,categorical,Cramer's V,0.366,True,True,0.0 (underflow)
4,age,numeric,absolute rank-biserial,0.365,True,True,0.0 (underflow)
5,occupation,categorical,Cramer's V,0.350,True,True,0.0 (underflow)
6,hours-per-week,numeric,absolute rank-biserial,0.344,True,True,0.0 (underflow)
7,sex,categorical,Cramer's V,0.215,True,True,0.0 (underflow)
8,workclass,categorical,Cramer's V,0.182,True,True,0.0 (underflow)
9,capital-gain,numeric,absolute rank-biserial,0.179,True,True,0.0 (underflow)


,feature,missing rate,observed rate,difference (pp),Newcombe 95% CI (pp),missing n
0,workclass,0.0947,0.2481,-15.34,"(-16.44, -14.13)",2799
1,occupation,0.0943,0.2481,-15.38,"(-16.48, -14.18)",2809
2,native-country,0.2567,0.2390,1.77,"(-1.06, 4.83)",857


{'n': 48842, 'high_income_prevalence': 0.23928176569346055, 'fnlwgt_weighted_prevalence_descriptive_only': 0.2377759609645643, 'feature_tests': 14, 'BH_significant': 13, 'Bonferroni_significant': 13, 'AUC': 0.8989270610642558, 'AUC_bootstrap_95_CI': [0.8931921538635177, 0.9050196293956572], 'saved_run_exit_code': 0}
